In [4]:
import requests
import json
from hdfs import InsecureClient

#API urls
FX_URL = "https://www.cbn.gov.ng/api/GetAllExchangeRates"
INF_URL = "https://www.cbn.gov.ng/api/GetAllInflationRates"


# HDFS Connection 
hdfs_client = InsecureClient('http://hdfs-namenode:9870', user='root')
HDFS_LANDING_PATH = '/cbn_project/landing'

        
def ingest_cbn_data(url, hdfs_filename):
    # Ensure the directory exists, to create if it doesnt exist.
    if hdfs_client.status(HDFS_LANDING_PATH, strict=False) is None:
        print(f"Creating directory: {HDFS_LANDING_PATH}")
        hdfs_client.makedirs(HDFS_LANDING_PATH)
    
    print(f"Connecting to CBN API for {hdfs_filename}...")
    
    try:
        # (Rest of your logic remains the same)
        response = requests.get(url, timeout=30)
        response.raise_for_status() 
        
        data = response.json()
        hdfs_full_path = f"{HDFS_LANDING_PATH}/{hdfs_filename}"
        
        with hdfs_client.write(hdfs_full_path, encoding='utf-8', overwrite=True) as writer:
            json.dump(data, writer)
            
        print(f"Data Ingested succssfully to: {hdfs_full_path}")
        
    except Exception as e:
        print(f"Failed to ingest data: {e}")

# --- Execute ---
ingest_cbn_data(FX_URL, "raw_exchange_rates.json")
ingest_cbn_data(INF_URL, "raw_inflation_rates.json")

Creating directory: /cbn_project/landing
Connecting to CBN API for raw_exchange_rates.json...
Data Ingested succssfully to: /cbn_project/landing/raw_exchange_rates.json
Connecting to CBN API for raw_inflation_rates.json...
Data Ingested succssfully to: /cbn_project/landing/raw_inflation_rates.json
